# Airbnb Mexico City — Price Analysis
## Notebook 01: Data Acquisition & First Exploration

**Project:** What factors determine the nightly price of an Airbnb listing in Mexico City?  
**Author:** [Your Name]  
**Data source:** [Inside Airbnb](https://insideairbnb.com/get-the-data/) — Mexico City  
**Last updated:** May 2026

---

### Objective
This notebook covers the first phase of the project:
1. Download the dataset directly from Inside Airbnb
2. Load and inspect the main files
3. Perform an initial data quality assessment
4. Document findings to guide the cleaning phase

> **Note:** All subsequent analysis (cleaning, EDA, modeling) will be handled in separate notebooks to keep each step reproducible and easy to follow.

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import requests
import os
import warnings

warnings.filterwarnings('ignore')

# Plot styling
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
    'font.family': 'sans-serif'
})

SEED = 42
DATA_DIR = 'data/raw'
os.makedirs(DATA_DIR, exist_ok=True)

print('Libraries loaded successfully.')

## 2. Data Download

Inside Airbnb provides three main files for each city:

| File | Description |
|---|---|
| `listings.csv` | One row per listing — price, location, amenities, host info, reviews |
| `calendar.csv` | Daily availability and price for each listing (365 days) |
| `reviews.csv` | Individual guest reviews with date and text |

For this project we will focus on `listings.csv` as the primary source for price analysis. `calendar.csv` will be used later to study price seasonality.

In [ ]:
# Inside Airbnb — Mexico City file URLs
# Go to https://insideairbnb.com/get-the-data/ to verify latest URLs
# and replace below with the most recent scrape date available.

BASE_URL = 'https://data.insideairbnb.com/mexico/df/mexico-city'

# Update SCRAPE_DATE to the most recent date shown on Inside Airbnb
SCRAPE_DATE = '2024-12-19'  # <-- update if needed

FILES = {
    'listings':  f'{BASE_URL}/{SCRAPE_DATE}/data/listings.csv.gz',
    'calendar':  f'{BASE_URL}/{SCRAPE_DATE}/data/calendar.csv.gz',
    'reviews':   f'{BASE_URL}/{SCRAPE_DATE}/data/reviews.csv.gz',
}

def download_file(url: str, dest_path: str) -> None:
    """Download a file from a URL and save it locally."""
    if os.path.exists(dest_path):
        print(f'  Already exists, skipping: {dest_path}')
        return
    print(f'  Downloading {os.path.basename(dest_path)}...')
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    with open(dest_path, 'wb') as f:
        f.write(response.content)
    size_mb = os.path.getsize(dest_path) / 1_000_000
    print(f'  Saved ({size_mb:.1f} MB): {dest_path}')


print('Downloading Inside Airbnb — Mexico City datasets...')
for name, url in FILES.items():
    dest = os.path.join(DATA_DIR, f'{name}.csv.gz')
    download_file(url, dest)

print('\nAll files ready.')

> **Manual download option:** If the URL above is outdated, visit [insideairbnb.com/get-the-data](https://insideairbnb.com/get-the-data/), find *Mexico City*, download the files manually, and place them in `data/raw/`.

## 3. Load the Listings Dataset

In [ ]:
listings_path = os.path.join(DATA_DIR, 'listings.csv.gz')
df = pd.read_csv(listings_path, low_memory=False)

print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head(3)

## 4. Initial Inspection

### 4.1 Column overview

In [ ]:
# Quick view of all columns, their data types and example values
col_summary = pd.DataFrame({
    'dtype':       df.dtypes,
    'non_null':    df.notna().sum(),
    'null_pct':    (df.isna().mean() * 100).round(1),
    'example':     df.iloc[0]
})

# Show only columns with meaningful data (skip near-empty ones)
col_summary.sort_values('null_pct')

### 4.2 Columns selected for analysis

Out of the ~70+ available columns, we will focus on the following based on their relevance to price prediction and business interpretation:

| Column | Type | Relevance |
|---|---|---|
| `price` | string → float | **Target variable** |
| `neighbourhood_cleansed` | categorical | Location (colonia) |
| `room_type` | categorical | Entire home vs private room |
| `accommodates` | integer | Capacity |
| `bedrooms` | float | Number of bedrooms |
| `bathrooms_text` | string | Number of bathrooms |
| `minimum_nights` | integer | Booking constraint |
| `number_of_reviews` | integer | Popularity signal |
| `review_scores_rating` | float | Quality signal |
| `host_is_superhost` | boolean | Host quality |
| `host_listings_count` | integer | Host experience |
| `amenities` | string (list) | Features included |
| `latitude` / `longitude` | float | Geospatial position |

In [ ]:
COLS_OF_INTEREST = [
    'id', 'name',
    'neighbourhood_cleansed',
    'latitude', 'longitude',
    'room_type',
    'accommodates',
    'bedrooms', 'bathrooms_text',
    'price',
    'minimum_nights',
    'number_of_reviews',
    'review_scores_rating',
    'host_is_superhost',
    'host_listings_count',
    'amenities',
    'availability_365'
]

df_core = df[COLS_OF_INTEREST].copy()
print(f'Working dataset: {df_core.shape[0]:,} rows × {df_core.shape[1]} columns')
df_core.head()

### 4.3 Missing values

In [ ]:
missing = (
    df_core.isna().mean() * 100
).round(2).sort_values(ascending=False)

missing = missing[missing > 0]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(missing.index, missing.values, color='#378ADD', height=0.6)
ax.bar_label(bars, fmt='%.1f%%', padding=4, fontsize=10)
ax.set_xlabel('Missing values (%)')
ax.set_title('Missing data by column — listings.csv')
ax.set_xlim(0, missing.max() * 1.2)
plt.tight_layout()
plt.show()

print('\nColumns with no missing values:')
print([c for c in COLS_OF_INTEREST if c not in missing.index])

### 4.4 Target variable: price

In [ ]:
# The price column comes as a string like '$1,200.00' — inspect raw values first
print('Raw price examples:')
print(df_core['price'].dropna().head(10).tolist())

print(f'\nNull prices: {df_core["price"].isna().sum():,}')
print(f'Zero prices: {(df_core["price"] == "$0.00").sum():,}')

In [ ]:
# Convert price to float
df_core['price_usd'] = (
    df_core['price']
    .str.replace(r'[$,]', '', regex=True)
    .astype(float)
)

print('Price statistics (USD):')
print(df_core['price_usd'].describe().round(2))

In [ ]:
# Price distribution — raw (before cleaning outliers)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

price_filtered = df_core.query('price_usd > 0 & price_usd < 1000')['price_usd']

# Histogram
axes[0].hist(price_filtered, bins=60, color='#378ADD', edgecolor='white', linewidth=0.3)
axes[0].axvline(price_filtered.mean(),   color='#E24B4A', linestyle='--', linewidth=1.5, label=f'Mean: ${price_filtered.mean():.0f}')
axes[0].axvline(price_filtered.median(), color='#1D9E75', linestyle='--', linewidth=1.5, label=f'Median: ${price_filtered.median():.0f}')
axes[0].set_xlabel('Price per night (USD)')
axes[0].set_ylabel('Number of listings')
axes[0].set_title('Price distribution (< $1,000)')
axes[0].legend(fontsize=10)

# Log-scale histogram (more informative for skewed distributions)
log_prices = np.log1p(df_core.query('price_usd > 0')['price_usd'])
axes[1].hist(log_prices, bins=60, color='#1D9E75', edgecolor='white', linewidth=0.3)
axes[1].set_xlabel('log(1 + price) — USD')
axes[1].set_ylabel('Number of listings')
axes[1].set_title('Price distribution (log scale)')

plt.suptitle('Airbnb Mexico City — Nightly Price Distribution', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('data/raw/price_distribution_raw.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'\nNote: The distribution is right-skewed (mean > median).')
print(f'This is expected — a few luxury listings pull the mean upward.')
print(f'We will use log(price) as the target variable in the regression model.')

### 4.5 Categorical variables

In [ ]:
# Room type distribution
room_counts = df_core['room_type'].value_counts()

fig, ax = plt.subplots(figsize=(7, 3.5))
colors = ['#378ADD', '#1D9E75', '#EF9F27', '#E24B4A']
bars = ax.barh(room_counts.index, room_counts.values, color=colors[:len(room_counts)], height=0.5)
ax.bar_label(bars, fmt='{:,.0f}', padding=4, fontsize=10)
ax.set_xlabel('Number of listings')
ax.set_title('Listings by room type — Mexico City')
ax.set_xlim(0, room_counts.max() * 1.15)
plt.tight_layout()
plt.show()

print(room_counts.to_frame('count').assign(pct=lambda x: (x['count'] / x['count'].sum() * 100).round(1)))

In [ ]:
# Top 20 neighbourhoods by listing count
top_neighbourhoods = df_core['neighbourhood_cleansed'].value_counts().head(20)

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(top_neighbourhoods.index[::-1], top_neighbourhoods.values[::-1], color='#378ADD', height=0.6)
ax.set_xlabel('Number of listings')
ax.set_title('Top 20 neighbourhoods by listing count')
plt.tight_layout()
plt.show()

### 4.6 Numeric variables — quick overview

In [ ]:
numeric_cols = ['accommodates', 'bedrooms', 'minimum_nights',
                'number_of_reviews', 'review_scores_rating',
                'host_listings_count', 'availability_365', 'price_usd']

df_core[numeric_cols].describe().round(2)

## 5. Data Quality Summary

Based on the inspection above, here are the main issues to address in the **cleaning notebook** (`02_data_cleaning.ipynb`):

| Issue | Column(s) | Action planned |
|---|---|---|
| Price stored as string | `price` | Strip `$` and `,`, convert to float ✅ done above |
| Listings with `price = 0` | `price_usd` | Remove — not real listings |
| Extreme price outliers | `price_usd` | Cap or remove using IQR rule |
| Bathrooms as text | `bathrooms_text` | Extract numeric value with regex |
| Missing `bedrooms` | `bedrooms` | Impute with median by room type |
| Missing `review_scores_rating` | `review_scores_rating` | Impute with median or flag as 'no reviews' |
| `minimum_nights` extremes | `minimum_nights` | Listings > 30 nights are not short-term rentals — consider removing |
| `host_is_superhost` boolean | `host_is_superhost` | Convert `t/f` strings to `True/False` |
| `amenities` as JSON string | `amenities` | Parse and count for feature engineering |

**Right-skewed target variable:** As seen in the distribution plot, `price_usd` is heavily right-skewed. We will use `log(price_usd)` as the target variable in the regression model, which is standard practice for price data.

## 6. Save for Next Notebook

In [ ]:
os.makedirs('data/processed', exist_ok=True)

df_core.to_csv('data/processed/listings_core.csv', index=False)

print('Saved: data/processed/listings_core.csv')
print(f'Shape: {df_core.shape[0]:,} rows × {df_core.shape[1]} columns')
print('\nReady for Notebook 02: Data Cleaning')

---

## Key Takeaways

- The dataset contains **~18,000+ active listings** across Mexico City's neighbourhoods.
- The majority of listings are **entire homes/apartments**, which aligns with investor interest.
- Price is **right-skewed** — a log transformation will be needed before modeling.
- Several columns require cleaning (price parsing, bathrooms text, boolean flags).
- The data has **good coverage** for the features most relevant to price analysis.

**Next step →** `02_data_cleaning.ipynb`